In [ ]:
# ============================================================
# CELL 1: SETUP, TRAINING & VALIDATION EVALUATION
# ============================================================

import os
import json
import time
import pickle
import warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

# 1. CONFIGURATION
RANDOM_STATE = 42
ARTIFACT_DIR = "./ML_Artifacts"
OUTPUT_DIR = "./RandomForest_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. LOAD DATA
print("=" * 70)
print("LOADING DATA ARTIFACTS")
print("=" * 70)

X_train = sp.load_npz(os.path.join(ARTIFACT_DIR, "X_train_tfidf.npz"))
X_val   = sp.load_npz(os.path.join(ARTIFACT_DIR, "X_val_tfidf.npz"))
X_test  = sp.load_npz(os.path.join(ARTIFACT_DIR, "X_test_tfidf.npz"))

y_train = np.load(os.path.join(ARTIFACT_DIR, "y_train.npy"))
y_val   = np.load(os.path.join(ARTIFACT_DIR, "y_val.npy"))
y_test  = np.load(os.path.join(ARTIFACT_DIR, "y_test.npy"))

mapping_path = os.path.join(ARTIFACT_DIR, "class_mapping.json")
if os.path.exists(mapping_path):
    with open(mapping_path, "r", encoding="utf-8") as f:
        mapping_data = json.load(f)
    id_to_class = {int(k): v for k, v in mapping_data["id_to_class"].items()}
    class_names = [id_to_class[i] for i in range(len(id_to_class))]
else:
    class_names = ["SQLInjection", "XSS", "CommandInjection", "Normal"]

print(f"X_train : {X_train.shape} | y_train : {y_train.shape}")
print(f"X_val   : {X_val.shape}   | y_val   : {y_val.shape}")
print(f"X_test  : {X_test.shape}  | y_test  : {y_test.shape}")


# 3. REUSABLE EVALUATION FUNCTION
def evaluate_dataset(model, X, y, dataset_name):
    print("\n" + "=" * 70)
    print(f"RandomForest - {dataset_name}")
    print("=" * 70)

    start_time = time.time()
    y_pred = model.predict(X)
    pred_time = time.time() - start_time

    acc = accuracy_score(y, y_pred)
    prec_macro = precision_score(y, y_pred, average="macro", zero_division=0)
    rec_macro = recall_score(y, y_pred, average="macro", zero_division=0)
    f1_macro = f1_score(y, y_pred, average="macro", zero_division=0)
    f1_weighted = f1_score(y, y_pred, average="weighted", zero_division=0)

    print(f"Accuracy       : {acc:.4f}")
    print(f"Precision Macro: {prec_macro:.4f}")
    print(f"Recall Macro   : {rec_macro:.4f}")
    print(f"F1 Macro       : {f1_macro:.4f}")
    print(f"F1 Weighted    : {f1_weighted:.4f}")
    print(f"Prediction Time: {pred_time:.2f} seconds")

    # Classification Report
    report_dict = classification_report(
        y, y_pred, target_names=class_names, digits=6, zero_division=0, output_dict=True
    )
    report_df = pd.DataFrame(report_dict).transpose()
    print("\nClassification Report:")
    display(report_df)

    # Confusion Matrix
    cm = confusion_matrix(y, y_pred)
    print("\nConfusion Matrix:")
    print(cm)

    # Save Artifacts
    report_path = os.path.join(OUTPUT_DIR, f"RandomForest_{dataset_name}_ClassificationReport.csv")
    report_df.to_csv(report_path)

    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    cm_path = os.path.join(OUTPUT_DIR, f"RandomForest_{dataset_name}_ConfusionMatrix.csv")
    cm_df.to_csv(cm_path, encoding="utf-8-sig")

    cmd_idx = class_names.index("CommandInjection") if "CommandInjection" in class_names else 2
    cmd_recall = recall_score(y, y_pred, labels=[cmd_idx], average="macro", zero_division=0)
    cmd_f1 = f1_score(y, y_pred, labels=[cmd_idx], average="macro", zero_division=0)

    summary_metrics = {
        "Model": "RandomForest",
        "Dataset": dataset_name,
        "Accuracy": acc,
        "Precision_Macro": prec_macro,
        "Recall_Macro": rec_macro,
        "F1_Macro": f1_macro,
        "F1_Weighted": f1_weighted,
        "Command_Recall": cmd_recall,
        "Command_F1": cmd_f1,
        "Prediction_Time": pred_time
    }
    return summary_metrics


# 4. TRAIN MODEL
print("\n" + "=" * 70)
print("TRAINING RANDOM FOREST (16 THREADS)")
print("=" * 70)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start_train = time.time()
rf_model.fit(X_train, y_train)
rf_training_time = time.time() - start_train
print(f"Random Forest Training Time: {rf_training_time:.2f} seconds ({rf_training_time / 60:.2f} minutes)")

# Save Model
with open(os.path.join(OUTPUT_DIR, "RandomForest_Model.pkl"), "wb") as f:
    pickle.dump(rf_model, f)

# 5. RUN VALIDATION
val_result = evaluate_dataset(rf_model, X_val, y_val, "Validation")
val_result["Training_Time"] = rf_training_time

val_df = pd.DataFrame([val_result])
val_df.to_csv(os.path.join(OUTPUT_DIR, "RandomForest_Validation_Summary.csv"), index=False)
print(f"\n✓ บันทึกผลลัพธ์ Validation เรียบร้อยใน: {OUTPUT_DIR}")

LOADING DATA ARTIFACTS
X_train : (35000, 30000) | y_train : (35000,)
X_val   : (7500, 30000)   | y_val   : (7500,)
X_test  : (7500, 30000)  | y_test  : (7500,)

TRAINING RANDOM FOREST (16 THREADS)
Random Forest Training Time: 21.24 seconds (0.35 minutes)

RandomForest - Validation
Accuracy       : 0.9757
Precision Macro: 0.9767
Recall Macro   : 0.9757
F1 Macro       : 0.9758
F1 Weighted    : 0.9758
Prediction Time: 0.45 seconds

Classification Report:


,precision,recall,f1-score,support
SQLInjection,0.994073,0.984000,0.989011,1875.000000
XSS,0.997330,0.996267,0.996798,1875.000000
CommandInjection,0.987093,0.938133,0.961991,1875.000000
Normal,0.928105,0.984533,0.955487,1875.000000
accuracy,0.975733,0.975733,0.975733,0.975733
macro avg,0.976650,0.975733,0.975822,7500.000000
weighted avg,0.976650,0.975733,0.975822,7500.000000



Confusion Matrix:
[[1845    0    4   26]
 [   0 1868    5    2]
 [   1    0 1759  115]
 [  10    5   14 1846]]

✓ บันทึกผลลัพธ์ Validation เรียบร้อยใน: ./RandomForest_Results


In [ ]:
# ============================================================
# CELL 2: FINAL TEST EVALUATION
# ============================================================

print("=" * 70)
print("EVALUATING FINAL TEST SET")
print("=" * 70)

# รันประเมิน Test Set
test_result = evaluate_dataset(rf_model, X_test, y_test, "Test")
test_result["Training_Time"] = rf_training_time

# บันทึกสรุปผล Test Set แยกต่างหาก
test_df = pd.DataFrame([test_result])
test_df.to_csv(os.path.join(OUTPUT_DIR, "RandomForest_Test_Summary.csv"), index=False)

# แสดงตารางเปรียบเทียบ Val vs Test
comparison_df = pd.DataFrame([val_result, test_result])
print("\n" + "=" * 70)
print("VALIDATION VS TEST COMPARISON")
print("=" * 70)
display(comparison_df[["Dataset", "Accuracy", "F1_Macro", "Command_Recall", "Command_F1", "Prediction_Time"]].round(4))

print(f"\n✓ บันทึกผลลัพธ์ Test Set เรียบร้อยใน: {OUTPUT_DIR}")